# DLFT Contact & Friction in the Admittance / FBS Form
**`cdc8bfe → e8ded2e`**  ·  branch `vandcard_DLFT`  ·  02.06.2026

Three things on this branch: **(A)** the solver was split into composable strategies, **(B)** a
**Dynamic Lagrangian Frequency–Time (DLFT)** method was added for unilateral contact and Coulomb
friction (Vadcard et al., JSV 2022; Nacivet et al., JSV 2003), **(C)** three examples validate it.

| File | Change |
|---|---|
| `frf_provider.py` | **new** — `FRFProvider`, `NumericalFRF`, `ExperimentalFRF` |
| `nonlinear_method.py` | **new** — `NonlinearMethod`, `AFT`, `DLFTContact`, `DLFTFriction` |
| `hbm_problems.py` | **new** — `FRFProblem`, `FBSProblem` |
| `frequency_domain.py` | cache slots; DC-bin Jacobian fix; transitional `FBS_DLFT` |
| `numerical_continuation/` | Newton line-search, rel/stagnation tol; `TangentPredictorBordered` |
| `examples/` | SDOF vibro-impact, FE rod, beam friction damper |

## Notation  (capital = frequency domain, lowercase = time)

| | |
|---|---|
| $\mathbf{Q}_r,\;\mathbf{q}_r(t)$ | relative interface displacement — the Newton unknown, $\mathbf{Q}_r=\mathbf{B}\mathbf{Q}$ |
| $\boldsymbol{\Lambda},\;\boldsymbol{\lambda}(t)$ | nonlinear / contact force (the *dynamic Lagrangian*) |
| $\mathbf{Y}=\mathbf{Z}^{-1}$ | FRF / admittance (block-diagonal over harmonics) |
| $\mathbf{Y}_r=\mathbf{B}\mathbf{Y}\mathbf{B}^{\!\top},\;\mathbf{Z}_r=\mathbf{Y}_r^{-1}$ | interface admittance / stiffness |
| $\mathbf{R}$ | residue · $\mathbf{B}$ Boolean coupling · $\mathbf{F}^{adm}=\mathbf{B}\mathbf{Y}\mathbf{F}^{ext}$ free response |
| $\mathrm{DFT}\{\cdot\},\,\mathrm{iDFT}\{\cdot\}$ | discrete Fourier transform / inverse · $\varepsilon$ penalty · $g_0$ gap · $\mu$ friction · $(\cdot)_{RI}$ stacked real form |

---
# Part A — Solver structure

AFT used to be hard-coded inside the FBS classes. It is now split into **three composable strategies**:

$$\Large \underbrace{\text{FRFProvider}}_{\mathbf{Y},\,\partial_\omega\mathbf{Y}}\;+\;\underbrace{\text{NonlinearMethod}}_{\boldsymbol{\Lambda},\,\partial\boldsymbol{\Lambda}}\;\longrightarrow\;\underbrace{\text{FRFProblem / FBSProblem}}_{\mathbf{R},\,\partial\mathbf{R}}$$

The continuation engine is unchanged — it still only needs `compute_residue_RI`,
`compute_jacobian_of_residue_RI`, `compute_derivative_wrt_omega_RI`. **AFT and DLFT are now interchangeable.**

## A.1 · `FRFProvider` — the admittance $\mathbf{Y}(\omega)$

`NumericalFRF` builds it analytically from $\mathbf{M},\mathbf{C},\mathbf{K}$; `ExperimentalFRF` splines
measured FRF data (the pyFBS hook).

$$\Large \mathbf{Z}_n = -(n\omega)^2\mathbf{M} + \mathrm{i}\,n\omega\,\mathbf{C} + \mathbf{K},\qquad \mathbf{Y}_n=\mathbf{Z}_n^{-1}$$

$$\Large \frac{\partial \mathbf{Y}_n}{\partial\omega}=-\mathbf{Y}_n\,\frac{\partial\mathbf{Z}_n}{\partial\omega}\,\mathbf{Y}_n,\qquad \frac{\partial\mathbf{Z}_n}{\partial\omega}=-2n^2\omega\,\mathbf{M}+\mathrm{i}\,n\,\mathbf{C}$$

## A.2 · `NonlinearMethod` — the force $\boldsymbol{\Lambda}$

Supplies $\boldsymbol{\Lambda}$ and its two derivatives in RI form. The `bind(problem)` hook lets a
contact method read problem-level data ($\mathbf{B}$, $\mathbf{Y}$, $\mathbf{F}^{ext}$).

| Method | Nonlinearity | $\boldsymbol{\Lambda}$ | unknown |
|---|---|---|---|
| `AFT` | smooth $f_{nl}(q,\dot q)$ | $\mathrm{DFT}\{f_{nl}(\mathrm{iDFT}\{\mathbf{Q}\})\}$ | $\mathbf{Q}$ |
| `DLFTContact` | unilateral normal | $\max(0,\lambda_p)$ predict–correct | $\mathbf{Q}_r$ |
| `DLFTFriction` | Coulomb + unilateral | stick/slip/separation sweep | $\mathbf{Q}_r$ |

## A.3 · `FRFProblem` / `FBSProblem` — the residue

**`FRFProblem`** (single subsystem):

$$\Large \boxed{\;\mathbf{R}=\mathbf{Q}+\mathbf{Y}\,(\boldsymbol{\Lambda}-\mathbf{F}^{ext})\;}$$

**`FBSProblem`** (FBS, unknown = relative interface DOF $\mathbf{Q}_r$, with $\mathbf{B}=\mathbf{I}_{N_h}\!\otimes\mathbf{B}_c$):

$$\Large \boxed{\;\mathbf{R}=\mathbf{Q}_r+\mathbf{B}\,\mathbf{Y}\,\mathbf{B}^{\!\top}\boldsymbol{\Lambda}-\mathbf{B}\,\mathbf{Y}\,\mathbf{F}^{ext}\;}$$

writing $\;\mathbf{Y}_r=\mathbf{B}\,\mathbf{Y}\,\mathbf{B}^{\!\top}\;$ and $\;\mathbf{F}^{adm}=\mathbf{B}\,\mathbf{Y}\,\mathbf{F}^{ext}\;$ for short. Jacobian and $\omega$-derivative ($\mathbf{Y}\!\to\!\mathbf{Y}_r$ for FBS):

$$\frac{\partial\mathbf{R}}{\partial\mathbf{Q}_r}=\mathbf{I}+(\mathbf{Y}_r)_{RI}\left.\frac{\partial\boldsymbol{\Lambda}}{\partial\mathbf{Q}_r}\right|_{RI},\qquad \frac{\partial\mathbf{R}}{\partial\omega}=\mathbf{B}\frac{\partial\mathbf{Y}}{\partial\omega}\big(\mathbf{B}^{\!\top}\boldsymbol{\Lambda}-\mathbf{F}^{ext}\big)\bigg|_{RI}+(\mathbf{Y}_r)_{RI}\left.\frac{\partial\boldsymbol{\Lambda}}{\partial\omega}\right|_{RI}$$

`compute_full_response` recovers all $d_{total}$ DOFs via $\mathbf{Q}=\mathbf{Y}(\mathbf{F}^{ext}-\mathbf{B}^{\!\top}\boldsymbol{\Lambda})$.

## A.4 · Caches & continuation

- **`FourierOmegaPoint` caches** the blocks shared by residue/Jacobian/$\partial_\omega$: `Y_cache`,
  `BY_cache`, `BYBT_RI_cache`, `Yr_cache`, `Zr_rhs`, `lambda_corrected`, `contact_mask`.
- **Continuation:** Newton backtracking line-search + relative/stagnation tolerances; parameterization
  kwargs unified; `TangentPredictorRobust` no longer double-scales by the step length.
- **`TangentPredictorBordered`** (Keller) — tangent from the bordered system, stable at limit points:

$$\begin{bmatrix}\partial_{\mathbf{Q}_r}\mathbf{R} & \partial_\omega\mathbf{R}\\[2pt]\dot{\mathbf{t}}_{\text{prev}}^{\!\top}\end{bmatrix}\mathbf{t}=\begin{bmatrix}\mathbf{0}\\1\end{bmatrix}$$

## A.5 · DC-harmonic Jacobian fix (`JacobianFourier_Real`)

The AFT analytic Jacobian maps the **time-domain** tangent $\mathbf{g}(t)=\partial f_{nl}/\partial q$ to the
**frequency-domain** block Jacobian through a Hankel/Toeplitz structure: the block coupling force-harmonic
$n$ to displacement-harmonic $m$ is the sum of the $(n\!-\!m)$-th and $(n\!+\!m)$-th Fourier coefficients of
$\mathbf{g}$,

$$\Large [\,\mathbf{J}\,]_{n,m}\;\propto\;\mathbf{G}_{\,n-m}+\mathbf{G}_{\,n+m}.$$

This doubling is right for $m\ge1$ but **wrong for the DC column** $m=0$. With pyhbm's real `rfft`
convention the spectral coefficients are

$$\Large \mathbf{c}_0 = N_t\,\mathbf{a}_0,\qquad \mathbf{c}_k=\tfrac{N_t}{2}\big(\mathbf{a}_k-\mathrm{i}\,\mathbf{b}_k\big),\;\;k\ge1,$$

i.e. **the DC bin carries no factor of 2.** For $m=0$ the two Hankel terms coincide
($\mathbf{G}_{n-0}=\mathbf{G}_{n+0}=\mathbf{G}_n$), so the naive sum counts $\mathbf{G}_n$ **twice** — the
$m=0$ Jacobian column is over-predicted by exactly $2\times$. The fix halves that one column:

```python
if 0 in Fourier.harmonics:
    m0 = list(Fourier.harmonics).index(0)
    plus[:, m0, :, :]  *= 0.5
    minus[:, m0, :, :] *= 0.5   # already zero by construction; kept for symmetry
```

**Why it was latent until DLFT.** A smooth nonlinearity rarely has a large static (DC) response, so the
mis-scaled $m=0$ column barely mattered. A vibro-impact response, however, has a big static offset (the
mass rests against the wall), so harmonic $0$ dominates and the error broke Newton convergence. It surfaced
in the SDOF example's finite-difference Jacobian check, where the disagreement sat entirely in the $m=0$
column. The correction lives in the **shared** AFT machinery, so it also tightens the DLFT contact/friction
Jacobian (which reuses `JacobianFourier_Real` for the masked operator, B.2 / B.4).

---
# Part B — DLFT

DLFT computes the periodic response of systems with unilateral contact / dry friction **without
artificial contact springs**: the contact forces come straight from the equations of motion, with the
time-domain non-penetration / Coulomb conditions injected through a penalised difference. We present the
**unilateral contact** case first (Vadcard vibro-impact), then the **Coulomb friction** generalisation
(Nacivet).

## B.1 · Unilateral contact — `DLFTContact` (Vadcard vibro-impact)

Interface admittance and free relative response, assembled with the Boolean matrix $\mathbf{B}$:

$$\Large \mathbf{Y}_r=\mathbf{B}\,\mathbf{Y}\,\mathbf{B}^{\!\top},\qquad \mathbf{F}^{adm}=\mathbf{B}\,\mathbf{Y}\,\mathbf{F}^{ext}$$

The Newton unknown is $\mathbf{Q}_r$ (residue = the `FBSProblem` residue of A.3). Per residue evaluation
(contact when $x_r>g_0$, one-signed force) — **predict** in time, **correct** with $\max(0,\cdot)$,
**re-project** with the DFT:

$$\Large \boldsymbol{\lambda}_p(t)=\mathrm{iDFT}\big\{\,\mathbf{Y}_r^{-1}\,(\mathbf{F}^{adm}-\mathbf{Q}_r)\,\big\}+\varepsilon\,\big(\mathbf{q}_r(t)-g_0\big)$$

$$\Large \boldsymbol{\lambda}(t)=\max\!\big(0,\,\boldsymbol{\lambda}_p(t)\big),\qquad \boldsymbol{\Lambda}=\mathrm{DFT}\{\boldsymbol{\lambda}(t)\}$$

The $\max(0,\cdot)$ enforces non-penetration **and** selects the contact phase (an output, not an
assumption); the mask $\mathbf{m}(t)=\mathbb{1}[\boldsymbol{\lambda}_p>0]$ is cached for the Jacobian. This
is the dynamic Lagrangian (Nacivet Eq. 17, B.3) specialised to unilateral contact.

```python
def _get_Zr_rhs(self, x):
    if x.Zr_rhs is None:
        Yr        = self._get_Yr(x)
        Fext_admr = self._get_Fext_admr(x)
        x.Zr_rhs  = np.linalg.solve(Yr, Fext_admr - vstack(x.fourier.coefficients))
    return x.Zr_rhs

def _get_lambda_corrected(self, x):
    if x.lambda_corrected is None:
        n_int = self._problem.ode.dimension
        zr_fourier = Fourier(
            self._get_Zr_rhs(x).reshape(Fourier.number_of_harmonics, n_int, 1)
        )
        Fourier_Real.compute_time_series(zr_fourier)
        zr_t   = zr_fourier.time_series
        Fourier_Real.compute_time_series(x.fourier)
        q_rel  = x.fourier.time_series
        lambda_p = zr_t + self.epsilon * (q_rel - self.g_zero)
        x.contact_mask     = lambda_p > 0.0
        lambda_t_corr      = np.where(x.contact_mask, lambda_p, 0.0)
        lambda_x_corr      = Fourier_Real.new_from_time_series(lambda_t_corr)
        x.lambda_corrected = vstack(lambda_x_corr.coefficients)
    return x.lambda_corrected
```

> $\mathbf{Y}_r$ folds in substructure compliance automatically: in the rod example the flexible wall is a
> second substructure, its stiffness $k_{obs}$ lives entirely inside $\mathbf{Y}_r$, and the rigid wall is
> $k_{obs}\to\infty$ — **with no change to the DLFT code.** The converged solution is
> $\varepsilon$-independent (Vadcard 2022); the examples sweep $\varepsilon$ soft→stiff to stay in the basin.

## B.2 · `DLFTContact` — Jacobian & $\omega$-derivative

$$\Large \boxed{\;\frac{\partial\boldsymbol{\Lambda}}{\partial\mathbf{Q}_r}=\mathrm{DFT}\big\{\,\mathrm{diag}(\mathbf{m})\;\mathrm{iDFT}\big\{\,\varepsilon\mathbf{I}-\mathbf{Y}_r^{-1}\,\big\}\,\big\}\;}$$

The masked round-trip $\mathrm{DFT}\{\mathrm{diag}(\mathbf{m})\,\mathrm{iDFT}\{\cdot\}\}$ is **real-linear, not
holomorphic** → built with the AFT machinery `JacobianFourier_Real` (the contact mask is the time tangent of
$\max(0,\cdot)$); it must **not** be wrapped by `FRF_to_RI`. The complex-linear factor
$\varepsilon\mathbf{I}-\mathbf{Y}_r^{-1}$ **is** holomorphic → uses `FRF_to_RI`. Both the Jacobian and
$\partial_\omega\boldsymbol{\Lambda}$ are FD-verified in the SDOF example.

```python
def compute_J_int_RI(self, x: FourierOmegaPoint, ode) -> np.ndarray:
    self._get_lambda_corrected(x)                    # populates x.contact_mask
    n_int = ode.dimension
    contact_tangent = x.contact_mask * eye(n_int)    # (Nt, n_int, n_int)
    J_mask    = JacobianFourier_Real.new_from_time_series(contact_tangent)
    J_mask_RI = block([[J_mask.RR, J_mask.RI], [J_mask.IR, J_mask.II]])
    Yr = self._get_Yr(x)
    Zr = self._invert_Yr_blockwise(Yr, n_int)
    complex_int_dim = Fourier.number_of_harmonics * n_int
    M    = self.epsilon * eye(complex_int_dim) - Zr
    M_RI = block([[M.real, -M.imag], [M.imag, M.real]])
    return J_mask_RI @ M_RI
```

**$\omega$-derivative** (`compute_dF_int_domega_RI`, feeds the A.3 $\partial\mathbf{R}/\partial\omega$ column). At fixed $\mathbf{Q}_r$ only the frequency-domain part of the predictor depends on $\omega$ — through $\mathbf{Y}_r,\mathbf{F}^{adm}$; the $\varepsilon(\mathbf{q}_r-g_0)$ term is $\omega$-independent. So differentiate the **predicted** contact force $\boldsymbol{\lambda}_{\mathrm{pred}}=\mathbf{Y}_r^{-1}(\mathbf{F}^{adm}-\mathbf{Q}_r)$ (code: `Zr_rhs`) at fixed $\mathbf{Q}_r$, then apply the same correction → the **corrected** derivative is $\partial\boldsymbol{\Lambda}/\partial\omega$:

$$\Large \frac{\partial\boldsymbol{\Lambda}}{\partial\omega}=\frac{\partial\boldsymbol{\lambda}_{\mathrm{corr}}}{\partial\omega}=\mathrm{DFT}\Big\{\,\mathrm{diag}(\mathbf{m})\;\mathrm{iDFT}\Big\{\tfrac{\partial\boldsymbol{\lambda}_{\mathrm{pred}}}{\partial\omega}\Big\}\,\Big\}$$

$$\Large \frac{\partial\boldsymbol{\lambda}_{\mathrm{pred}}}{\partial\omega}=\mathbf{Y}_r^{-1}\,\mathbf{B}\frac{\partial\mathbf{Y}}{\partial\omega}\big(\mathbf{F}^{ext}-\mathbf{B}^{\!\top}\boldsymbol{\lambda}_{\mathrm{pred}}\big)$$

In the code: `dlambda_pred` $=\partial\boldsymbol{\lambda}_{\mathrm{pred}}/\partial\omega$, masked into `dlambda_t_corr`, transformed back to `dlambda_corr` $=\partial\boldsymbol{\Lambda}/\partial\omega$.

```python
def compute_dF_int_domega_RI(self, x: FourierOmegaPoint, ode) -> np.ndarray:
    problem = self._problem
    Y   = problem._get_FRF(x)
    dY  = problem.frf_provider.compute_FRF_derivative(
              x.omega, Fourier.harmonics, problem.d_total, Y)
    BdY    = problem.B_fourier @ dY
    Yr     = self._get_Yr(x)
    Zr_rhs = self._get_Zr_rhs(x)
    n_int  = ode.dimension
    dlambda_pred = np.linalg.solve(
        Yr, BdY @ (problem.F_ext_full - problem.B_fourier.T @ Zr_rhs)
    )
    dlambda_pred_fourier = Fourier(
        dlambda_pred.reshape(Fourier.number_of_harmonics, n_int, 1)
    )
    Fourier_Real.compute_time_series(dlambda_pred_fourier)
    dlambda_t_corr = np.where(x.contact_mask, dlambda_pred_fourier.time_series, 0.0)
    dlambda_corr   = Fourier_Real.new_from_time_series(dlambda_t_corr)
    dlambda_v      = vstack(dlambda_corr.coefficients)
    return vstack((dlambda_v.real, dlambda_v.imag))
```

## B.3 · Coulomb friction — `DLFTFriction` (Nacivet Eqs. 17, 21–33)

Friction needs the **general** dynamic Lagrangian. With $\mathbf{X}_r$ the relative displacement *integrated
in time* from the contact forces (at convergence $\mathbf{X}_r=\mathbf{Q}_r$):

$$\Large \boxed{\;\boldsymbol{\Lambda}=\mathbf{F}_r-\mathbf{Z}_r\mathbf{Q}_r+\varepsilon\,(\mathbf{Q}_r-\mathbf{X}_r)\;}$$

Interface DOFs are grouped per node into $n_{dir}=1+n_{tan}$ blocks (index 0 = normal $N$, rest = tangential
$T$; $n_{tan}=2$ → 2-D friction disc), with per-direction penalties $\boldsymbol{\varepsilon}$. Predictor:

$$\Large \boldsymbol{\lambda}_u(t)=\mathrm{iDFT}\big\{\,\mathbf{Y}_r^{-1}(\mathbf{F}^{adm}-\mathbf{Q}_r)\,\big\}+\boldsymbol{\varepsilon}\odot\mathbf{q}_r(t),\qquad \lambda_u^{N}\mathrel{-}=\varepsilon_N g_0$$

Friction is path-dependent, so the corrector is a **sequential forward sweep** over time samples carrying
$\boldsymbol{\lambda}^{T}_{n-1}$. With $s=\lambda_u^{N}$ ($s>0$ ⇒ contact) and
$\mathbf{p}=\boldsymbol{\lambda}_u^{T}-\boldsymbol{\lambda}^{T}_{n-1}$:

$$\Large \begin{aligned}
&\textbf{separation }(s\le0): &&\lambda^N=0,\;\;\boldsymbol{\lambda}^T=\mathbf{0}\\
&\textbf{stick }(\lVert\mathbf{p}\rVert<\mu s): &&\lambda^N=s,\;\;\boldsymbol{\lambda}^T=\mathbf{p}\\
&\textbf{slip }(\lVert\mathbf{p}\rVert\ge\mu s): &&\lambda^N=s,\;\;\boldsymbol{\lambda}^T=\mu s\,\hat{\mathbf{p}}
\end{aligned}$$

```python
def _get_lambda_corrected(self, x):
    if x.lambda_corrected is None:
        n_int   = self._problem.ode.dimension
        eps_vec = self._get_eps_vec(n_int)

        zr_fourier = Fourier(
            self._get_Zr_rhs(x).reshape(Fourier.number_of_harmonics, n_int, 1)
        )
        Fourier_Real.compute_time_series(zr_fourier)
        zr_t = zr_fourier.time_series                 # (Nt, n_int, 1)
        Fourier_Real.compute_time_series(x.fourier)
        q_rel = x.fourier.time_series                 # (Nt, n_int, 1)

        # predictor:  lambda_u = IDFT[z] + eps*u_r,  normal slot -= eps_N*g0
        lambda_u = zr_t + eps_vec.reshape(1, n_int, 1) * q_rel
        lambda_u[:, ::self.n_dir, 0] -= self.epsilon_N * self.g_zero

        lam, Jloc = self._corrector_sweep(lambda_u)
        x.contact_mask = Jloc                         # cache per-sample tangent
        lambda_corr    = Fourier_Real.new_from_time_series(lam)
        x.lambda_corrected = vstack(lambda_corr.coefficients)
    return x.lambda_corrected
```

The sequential stick/slip/separation corrector (the core of `_corrector_sweep`, looping contacts × sweeps × time samples):

```python
for c in range(n_contacts):
    nN = c * n_dir                      # normal slot
    sT = slice(nN + 1, nN + n_dir)      # tangential slots
    lamx_prev_T = zeros(n_tan)          # carried tangential state

    for _pass in range(self.n_sweep):
        lamx_start = lamx_prev_T.copy()
        for k in range(Nt):
            s = lu[k, nN]                       # predicted normal (>0 == contact)
            p = lu[k, sT] - lamx_prev_T         # predicted tangential
            if s <= 0.0:                        # SEPARATION
                lam[k, nN, 0]   = 0.0
                lam[k, sT, 0]   = 0.0
                lamx_prev_T     = lu[k, sT].copy()
                # Jloc block stays 0
            else:
                p_norm = np.sqrt(p @ p)
                if p_norm < mu * s:             # STICK
                    lam[k, nN, 0] = s
                    lam[k, sT, 0] = p
                    # lamx_prev_T unchanged
                    Jloc[k, nN, nN] = 1.0
                    Jloc[k, sT, sT] = I_t
                else:                           # SLIP
                    p_hat = p / p_norm
                    lam[k, nN, 0] = s
                    lam[k, sT, 0] = mu * s * p_hat
                    lamx_prev_T   +=  p * (1.0 - mu * s / p_norm)
                    Jloc[k, nN, nN] = 1.0
                    Jloc[k, sT, nN] = mu * p_hat
                    Jloc[k, sT, sT] = (mu * s / p_norm) * (I_t - np.outer(p_hat, p_hat))
        # periodicity check: state entering sample 0 == state leaving Nt-1
        if np.max(np.abs(lamx_prev_T - lamx_start)) < self.sweep_tol:
            break
return lam, Jloc
```

Default = **single sweep**, $\boldsymbol{\lambda}^{cor}_{-1}=\mathbf{0}$ (Nacivet Fig. 2); `n_sweep>1`
iterates toward an exactly-periodic tangential force (beyond the paper).

> **Sign convention:** normal force positive in compression (contact $s>0$, bound $\mu s$) — matches
> `DLFTContact`, opposite to Nacivet's $\mu|s|$ with $s<0$.

## B.4 · `DLFTFriction` — Jacobian (slip block) & $\omega$-derivative

The corrector also returns the block-diagonal per-sample tangent $J_{loc}$ (history coupling dropped).
Stick: $J_{loc}=\mathbf{I}$; separation: $J_{loc}=\mathbf{0}$; **slip:**

$$\Large \frac{\partial\lambda^N}{\partial\lambda_u^N}=1,\qquad \frac{\partial\boldsymbol{\lambda}^T}{\partial\lambda_u^N}=\mu\hat{\mathbf{p}},\qquad \frac{\partial\boldsymbol{\lambda}^T}{\partial\boldsymbol{\lambda}_u^T}=\frac{\mu s}{\lVert\mathbf{p}\rVert}\big(\mathbf{I}-\hat{\mathbf{p}}\hat{\mathbf{p}}^{\!\top}\big)$$

(the last block projects onto the friction circle — see the `Jloc[k, sT, sT]` line above). The interface
Jacobian then mirrors B.2 with $J_{loc}$ for the mask and $\boldsymbol{\varepsilon}=\mathrm{diag}(\varepsilon_N,\varepsilon_T,\dots)$ for $\varepsilon$:

```python
def compute_J_int_RI(self, x: FourierOmegaPoint, ode) -> np.ndarray:
    self._get_lambda_corrected(x)                    # populates x.contact_mask = Jloc(t)
    n_int = ode.dimension
    Jloc  = x.contact_mask                           # (Nt, n_int, n_int)
    J_mask    = JacobianFourier_Real.new_from_time_series(Jloc)
    J_mask_RI = block([[J_mask.RR, J_mask.RI], [J_mask.IR, J_mask.II]])
    Yr = self._get_Yr(x)
    Zr = self._invert_Yr_blockwise(Yr, n_int)
    eps_vec = self._get_eps_vec(n_int)
    E    = diag(np.tile(eps_vec, Fourier.number_of_harmonics))   # blkdiag(eps) over harmonics
    M    = E - Zr
    M_RI = block([[M.real, -M.imag], [M.imag, M.real]])
    return J_mask_RI @ M_RI
```

> Nacivet uses a **hybrid Powell** solver (robust through gross slip). `pyhbm` uses Newton + arc-length
> with the analytic tangent; in 1-D gross slip the $T\!\to\!T$ block vanishes, so heavy slip wants a
> Powell-type solver — the beam example runs in **partial slip** where Newton converges.

**$\omega$-derivative.** Same predicted-force sensitivity $\partial\boldsymbol{\lambda}_{\mathrm{pred}}/\partial\omega$ as `DLFTContact`, but the time-domain correction uses the per-sample block tangent $J_{loc}(t)$ instead of the scalar mask:

$$\Large \frac{\partial\boldsymbol{\Lambda}}{\partial\omega}=\mathrm{DFT}\Big\{\,J_{loc}\;\mathrm{iDFT}\Big\{\tfrac{\partial\boldsymbol{\lambda}_{\mathrm{pred}}}{\partial\omega}\Big\}\,\Big\},\qquad \frac{\partial\boldsymbol{\lambda}_{\mathrm{pred}}}{\partial\omega}=\mathbf{Y}_r^{-1}\,\mathbf{B}\frac{\partial\mathbf{Y}}{\partial\omega}\big(\mathbf{F}^{ext}-\mathbf{B}^{\!\top}\boldsymbol{\lambda}_{\mathrm{pred}}\big)$$

```python
def compute_dF_int_domega_RI(self, x: FourierOmegaPoint, ode) -> np.ndarray:
    self._get_lambda_corrected(x)                    # ensure x.contact_mask = Jloc(t)
    problem = self._problem
    Y   = problem._get_FRF(x)
    dY  = problem.frf_provider.compute_FRF_derivative(
              x.omega, Fourier.harmonics, problem.d_total, Y)
    BdY    = problem.B_fourier @ dY
    Yr     = self._get_Yr(x)
    Zr_rhs = self._get_Zr_rhs(x)
    n_int  = ode.dimension
    dlambda_pred = np.linalg.solve(
        Yr, BdY @ (problem.F_ext_full - problem.B_fourier.T @ Zr_rhs)
    )
    dlambda_pred_fourier = Fourier(
        dlambda_pred.reshape(Fourier.number_of_harmonics, n_int, 1)
    )
    Fourier_Real.compute_time_series(dlambda_pred_fourier)
    # per-sample block multiply by the cached contact tangent Jloc(t)
    Jloc = x.contact_mask                            # (Nt, n_int, n_int)
    dlambda_t_corr = einsum('kij,kjl->kil', Jloc, dlambda_pred_fourier.time_series)
    dlambda_corr   = Fourier_Real.new_from_time_series(dlambda_t_corr)
    dlambda_v      = vstack(dlambda_corr.coefficients)
    return vstack((dlambda_v.real, dlambda_v.imag))
```

---
# Part C — Examples

All built the same way: `provider = NumericalFRF(M,C,K)` · `method = DLFTContact/DLFTFriction(...)` ·
`problem = FBSProblem(system, provider, method)` · standard `HarmonicBalanceMethod` continuation.

## C.1 · SDOF vibro-impact vs. NLvib shooting
`examples/sdof_vibroimpact_validation/`

Mass–spring–damper hitting a wall ($q\le g_0$), $\mathbf{B}_c=[\,1\;\;-1\,]$, solved with `DLFTContact`.
Validates: FD check of $\partial\mathbf{R}/\partial\mathbf{Q}_r$ **and** $\partial\mathbf{R}/\partial\omega$
(this uncovered the DC-bin bug of A.5), and the FRC ($A_1$ + peak $|q|$) against an independent NLvib
shooting reference.

![SDOF DLFT vs NLvib](../examples/sdof_vibroimpact_validation/comparison_dlft_vs_nlvib.png)

## C.2 · FE rod against a flexible wall (Vadcard Fig. 17)
`examples/rod_example/`

Clamped–free axial bar (20 elements) impacting a **flexible obstacle** modelled as a *second substructure*
— grounded spring $k_{obs}=k_{rel}k_{rod}$, coupled only through DLFT on $x_r=u_B-u_w$. Reproduces the NFRC
for $k_{rel}=0.4,4,20,40$ vs. NLvib, plus a stiffness sweep warm-started up to a near-rigid branch.

![Rod NFRC vs NLvib](../examples/rod_example/rod_vibroimpact_frc.png)

![Rod flexible→rigid sweep](../examples/rod_example/rod_vibroimpact_rigid_vs_flexible.png)

## C.3 · Cantilever beam + dry-friction damper (Nacivet Figs. 4–5)
`examples/beam_friction_damper/`

The one example exercising **`DLFTFriction`**. Steel beam (2-D frame elements) with a damper node grounded
by springs $k_T,k_N$ and pre-loaded by $N_0$, contacting at $0.3L$ on relative coords
$[\,x_r^N;\,x_r^T\,]$. FD-verified; friction-damped, frequency-shifted resonance vs. the stuck-linear FRF.
Run in **partial slip** (Newton + analytic tangent), full FE model rather than the paper's reduced one.

![Beam + friction damper FRC](../examples/beam_friction_damper/beam_friction_damper_frc.png)

---
**Status:** the strategy API (`FRFProvider`/`NonlinearMethod`/`FRFProblem`/`FBSProblem`) is the path
forward; `FBS_DLFT` in `frequency_domain.py` is transitional. `DLFTFriction` is FD-verified but only the
beam example uses it (partial slip). Legacy frequency-domain classes are still exported pending cleanup.